# WaveResNet multi-stage SHA-256 refinement (Colab)

Run this end-to-end in a Colab GPU runtime. It clones the repo, installs deps (PyTorch + psann), 
generates a fresh dataset, trains the WaveResNet refiner across multiple stages, and plots validation
error vs. baseline. If you already have a GPU runtime, just run cells top-to-bottom.

**Tip:** Runtime -> Change runtime type -> GPU before running.


In [ ]:
# Quick GPU sanity check (safe to re-run)
import subprocess, sys

try:
    subprocess.run(['nvidia-smi'], check=True)
except FileNotFoundError:
    print('nvidia-smi not found; ensure you selected a GPU runtime.')
except subprocess.CalledProcessError as exc:
    print('nvidia-smi failed:', exc)


In [ ]:
# Install deps and clone the repo
import os, sys, subprocess, shutil
from pathlib import Path

REPO_URL = 'https://github.com/<your-username>/sha-fun.git'  # << update to your GitHub repo
PROJECT_DIR = Path('/content/sha-fun')

# Pin torch CUDA wheels so psann runs on GPU; adjust cu121->cu124 if Colab updates CUDA.
TORCH_VERSION = '2.4.1+cu121'
TORCHVISION_VERSION = '0.19.1+cu121'
TORCHAUDIO_VERSION = '2.4.1+cu121'
TORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu121'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    f'torch=={TORCH_VERSION}', f'torchvision=={TORCHVISION_VERSION}', f'torchaudio=={TORCHAUDIO_VERSION}',
    '--index-url', TORCH_INDEX_URL,
], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psann==0.10.17', 'scikit-learn', 'pandas', 'matplotlib', 'tqdm'], check=True)

if PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(PROJECT_DIR)], check=True)
os.chdir(PROJECT_DIR)
sys.path.append(str(PROJECT_DIR))
print('Now working in', os.getcwd())

import torch
print('Torch version:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

# Optional: avoid too much torch/cublas logging in notebook
os.environ.setdefault('TORCH_SHOW_CPP_STACKTRACES', '0')
os.environ.setdefault('CUDA_LAUNCH_BLOCKING', '0')


In [ ]:
# Configure experiment hyperparameters
import random, time, json
from argparse import Namespace
from pathlib import Path

import numpy as np
import torch
from sklearn.model_selection import train_test_split

from data_generator import GenerateEvolutionMapDataset
from train_wave_resnet import build_estimator, evaluate_refinement, refine_dataset_with_model

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Keeps the run reasonably quick on Colab while still showing multi-stage improvements.
config = {
    'stages': 3,
    'num_samples': 4000,
    'min_len': 8,
    'max_len': 20,
    'noise_std': 0.2,
    'alphabet': None,  # None -> letters+digits+space (see generator)
    'epochs': 40,
    'batch_size': 64,
    'lr': 3e-4,
    'val_frac': 0.2,
    'conv_channels': 32,
    'kernel_size': 3,
    'hidden_layers': 4,
    'hidden_units': 96,
    'attention_heads': 0,
    'patience': 6,
    'early_stopping': True,
}

args = Namespace(
    out_dir=str(Path('runs') / 'wave_resnet'),
    stages=int(config['stages']),
    num_samples=int(config['num_samples']),
    min_len=int(config['min_len']),
    max_len=int(config['max_len']),
    noise_std=float(config['noise_std']),
    alphabet=config['alphabet'] or '',
    save_dataset='',
    epochs=int(config['epochs']),
    batch_size=int(config['batch_size']),
    lr=float(config['lr']),
    val_frac=float(config['val_frac']),
    conv_channels=int(config['conv_channels']),
    kernel_size=int(config['kernel_size']),
    hidden_layers=int(config['hidden_layers']),
    hidden_units=int(config['hidden_units']),
    attention_heads=int(config['attention_heads']),
    seed=int(SEED),
    device=device,
    patience=int(config['patience']),
    early_stopping=bool(config['early_stopping']),
    verbose=1,
)

print('Using device:', device)
print('Config:', json.dumps(config, indent=2))


In [ ]:
# Build a fresh dataset (stage 0 uses flat baseline=0.5)
inputs, residuals, targets = GenerateEvolutionMapDataset(
    num_samples=args.num_samples,
    min_len=args.min_len,
    max_len=args.max_len,
    alphabet=args.alphabet or None,
    noise_std=args.noise_std,
    seed=args.seed,
    save=False,
)

if inputs is None:
    raise RuntimeError('Dataset generation failed; try reducing num_samples or len range.')

print('Inputs:', inputs.shape, '| Residuals:', residuals.shape, '| Targets:', targets.shape)


In [ ]:
# Multi-stage WaveResNet training + evaluation
from pathlib import Path

timestamp = time.strftime('%Y%m%d-%H%M%S')
run_dir = Path(args.out_dir) / f'wave_resnet_{timestamp}'
run_dir.mkdir(parents=True, exist_ok=True)

stage_metrics = []
estimator = None

for stage in range(args.stages):
    if stage > 0:
        print(f"\n--- Building dataset for stage {stage} via previous model ---")
        inputs, residuals, targets = refine_dataset_with_model(estimator, inputs, targets)

    X_train, X_val, y_train, y_val, t_train, t_val = train_test_split(
        inputs, residuals, targets,
        test_size=args.val_frac,
        random_state=args.seed,
        shuffle=True,
    )

    estimator = build_estimator(args)
    print(
        f"Stage {stage}: training on {X_train.shape[0]} samples (val {X_val.shape[0]}), "
        f"epochs={args.epochs}, lr={args.lr}, device={args.device}"
    )
    estimator.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        verbose=int(args.verbose),
    )

    val_ratio, baseline_mae, refined_mae = evaluate_refinement(estimator, X_val, t_val)
    print(
        f"Stage {stage} validation relative MAE: {val_ratio:.4f} "
        f"(baseline {baseline_mae:.4f} -> refined {refined_mae:.4f})"
    )

    ckpt_path = run_dir / f'estimator_stage{stage}.pt'
    estimator.save(str(ckpt_path))
    stage_metrics.append({
        'stage': int(stage),
        'val_ratio': float(val_ratio),
        'baseline_mae': float(baseline_mae),
        'refined_mae': float(refined_mae),
        'checkpoint': ckpt_path.name,
    })

final_ckpt = run_dir / 'estimator.pt'
estimator.save(str(final_ckpt))

metrics_payload = {
    'stages': stage_metrics,
    'args': vars(args),
}
with open(run_dir / 'metrics.json', 'w', encoding='utf-8') as fh:
    json.dump(metrics_payload, fh, indent=2)

print(f"\nSaved checkpoints and metrics to {run_dir}")
stage_metrics


In [ ]:
# Plot stage-wise validation ratios (lower is better)
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(stage_metrics)
display(df)

plt.figure(figsize=(6, 4))
plt.plot(df['stage'], df['val_ratio'], marker='o', label='relative MAE')
plt.axhline(1.0, color='gray', linestyle='--', label='baseline = 1.0')
plt.xlabel('Stage')
plt.ylabel('Validation MAE / baseline MAE (lower is better)')
plt.title('WaveResNet refinement across stages')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
